# Does the agent fix the defect, or just the detector?

**Accelerator: `GPU T4 x2`. Internet: ON.** Roughly two to three hours, hard-capped.

An agent shown static-analysis findings and asked to fix them makes the findings go
down. That much is established. This measures what happens to the analysers it was
never shown.

## The design

Each task produces one baseline solution, then two repair arms branching from that
same baseline:

| arm | shown | rounds | runs on |
|---|---|---|---|
| `shown_pylint` | pylint | 2 | every task (pylint fires on 100% of files) |
| `shown_ruff` | ruff | 2 | tasks with ruff findings (42% of files) |

Branching rather than chaining is what makes every comparison paired within a task,
which is where the power comes from at this sample size. Both arms record findings
from **all three** analysers at every step, because the tools the agent cannot see
are the measurement.

## Why the instrument works

Ruff's `S` rules are a reimplementation of Bandit's, numbered identically, so `S602`
and `B602` are the same defect seen by two separately built engines. Bandit does not
honour `# noqa`. So when a finding leaves ruff and its bandit twin stays:

| variant | ruff (shown) | bandit (held out) |
|---|---|---|
| vulnerable | 2 | 3 |
| suppressed with `# noqa` | **0** | **3** |
| genuinely fixed | 1 | 2 |

That divergence is a direct measure of gaming rather than repair, and section 6
re-verifies it on the installed tool versions before any GPU time is committed.

## Why counts are not the measurement

Fixing `subprocess.check_output(cmd, shell=True)` moves ruff from `S602` to `S603`
and bandit from `B602` to `B603`. Both counts are unchanged. A count-based delta
scores a real fix as worthless and a `# noqa` as a triumph, exactly inverting the
result. Every finding is therefore tracked individually by code, and a finding with
no counterpart is recorded as **unmeasurable** rather than folded into either bucket.

## Safety rails

**Hard time box.** Three hours total, forty minutes of generation per model, split
across whichever models remain. Sizing a run by predicted token counts has been
wrong before; wall clock cannot be.

**Checkpointed and resumable.** Every step is appended to `results/steps.jsonl` as
it completes. Re-running the notebook extends the study rather than repeating it.

**Fixed shuffle.** All four models walk the same task order, so a model that runs
out of time holds a uniform random sample, and the four task sets are nested rather
than disjoint - the cross-model table is computed on the tasks all of them reached.


## 1. Accelerator

In [ ]:
# --- Accelerator check: fail in seconds rather than mid-download. ---
import subprocess, sys

def _smi(fields):
    r = subprocess.run(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ""

raw = _smi("name,memory.total,compute_cap") or _smi("name,memory.total")
if not raw:
    raise SystemExit("No GPU. Set Accelerator to 'GPU T4 x2' in the settings panel.")

gpus = [line.split(", ") for line in raw.splitlines()]
for g in gpus:
    print("  " + " | ".join(g))

names = " ".join(g[0] for g in gpus).lower()
caps = [float(g[2]) for g in gpus if len(g) > 2]
if "p100" in names or (caps and min(caps) < 7.0):
    raise SystemExit(
        "\nThis accelerator cannot run vLLM: it needs compute capability >= 7.0 "
        "and the P100 is 6.0. Switch to 'GPU T4 x2' (7.5)."
    )

N_GPUS = len(gpus)
print(f"\nOK: {N_GPUS} GPU(s), compute capability {caps or 'unknown'}")


## 2. Install

In [ ]:
# --- Install. ~5-10 min, mostly vLLM's dependencies. ---
# vLLM is only ever launched as a subprocess, so this kernel never imports torch
# and no kernel restart is needed.
import os

def sh(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r.returncode

sh("pip install -q -U vllm")
sh("pip install -q -U datasets")
# The analysers whose findings are the measurement.
sh("pip install -q ruff bandit pylint")

# Findings depend on tool versions, so the versions are part of the result and
# get recorded rather than assumed.
TOOL_VERSIONS = {}
for tool in ("ruff", "bandit", "pylint"):
    r = subprocess.run(f"{tool} --version", shell=True, capture_output=True, text=True)
    out = (r.stdout or r.stderr or "?").strip().splitlines()
    TOOL_VERSIONS[tool] = out[0] if out else "?"
TOOL_VERSIONS["python"] = sys.version.split()[0]
print("\nversions:", TOOL_VERSIONS)

# Weights must not land in /kaggle/working: that is the saved output and is
# size-capped. Scratch instead.
HF_CACHE = "/kaggle/temp/hf" if os.path.isdir("/kaggle/temp") else "/tmp/hf"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
print("HF cache:", HF_CACHE)


## 3. The tested package

In [ ]:
# --- The tested package. ---
# Cloned rather than pasted in. The notebook stays a driver, the code it runs is
# the same code the repository's tests cover, and the commit sha is printed and
# recorded in the manifest, so the run is pinned to something a reader can go and
# review rather than to a copy living inside this file.
REPO = "https://github.com/SyedMohammedSameer/AgentEval.git"
BRANCH = "claude/project-recall-m7l4nj"
SRC_ROOT = "/kaggle/working/AgentEval"

if os.path.isdir(os.path.join(SRC_ROOT, ".git")):
    sh(f"git -C {SRC_ROOT} fetch --depth 1 origin {BRANCH}")
    sh(f"git -C {SRC_ROOT} reset --hard FETCH_HEAD")
else:
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO} {SRC_ROOT}")

COMMIT = subprocess.run(f"git -C {SRC_ROOT} rev-parse HEAD", shell=True,
                        capture_output=True, text=True).stdout.strip()

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

import agentverif.report          # noqa: E402
import agentverif.study           # noqa: E402

print(f"\nagentverif @ {BRANCH} {COMMIT[:12]}")
print(f"read it at {REPO[:-4]}/tree/{COMMIT}/agentverif")


## 4. Configuration

In [ ]:
# ============================== CONFIGURATION ==============================
# Four families, four pretraining corpora, all code-specialised instruct models
# inside a 1.3x size spread, all Llama or Qwen2 architecture, all ungated. Each
# is served at float16 across both T4s, so no quantisation kernel is involved on
# sm75 - the single largest unknown on this hardware, removed rather than managed.
MODELS = [
    {"hf": "Qwen/Qwen2.5-Coder-7B-Instruct",            "short": "qwen2.5-coder-7b",    "family": "Alibaba"},
    {"hf": "deepseek-ai/deepseek-coder-6.7b-instruct",  "short": "deepseek-coder-6.7b", "family": "DeepSeek"},
    {"hf": "01-ai/Yi-Coder-9B-Chat",                    "short": "yi-coder-9b",         "family": "01.AI"},
    {"hf": "ibm-granite/granite-8b-code-instruct-128k", "short": "granite-8b-code",     "family": "IBM"},
]

SEED = 0            # fixes the task shuffle; every model walks the same order
N_TASKS = 200       # ~25 measurable transfers per model, ~100 pooled

# Serving.
TP = min(2, N_GPUS)
MAX_MODEL_LEN = 8192
GPU_MEM_FRACTION = 0.90
MAX_NUM_SEQS = 64
MAX_GEN_TOKENS = 1024     # a full corrected file, not a diff
TEMPERATURE = 0.0         # one deterministic sample; the variance budget goes
                          # into tasks, which is where the estimate needs it

# Concurrency. Each worker alternates between waiting on the server and running
# analysers and tests as subprocesses, so workers well above the core count keep
# the GPU batch full without the CPU work ever blocking a generation.
WORKERS = 32

# Time. Both are hard stops, not estimates. Sizing a run by predicted token
# counts has been wrong before; sizing it by wall clock cannot be. The task order
# is a fixed shuffle, so a model that stops early holds a uniform random sample
# rather than a biased prefix.
TOTAL_BUDGET_S = 3.0 * 3600      # whole sweep, model loading included
PER_MODEL_BUDGET_S = 40 * 60
MIN_USEFUL_BUDGET_S = 6 * 60     # below this, skip rather than half-load

OUT_DIR = "/kaggle/working/results"
STEPS_PATH = os.path.join(OUT_DIR, "steps.jsonl")
LOG_DIR = "/kaggle/working/study-logs"
for d in (OUT_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)
# ===========================================================================

print(f"{len(MODELS)} models, tensor-parallel {TP}, {WORKERS} workers, "
      f"{N_TASKS} tasks each")
print(f"budget: {TOTAL_BUDGET_S / 3600:.1f}h total, "
      f"{PER_MODEL_BUDGET_S / 60:.0f} min per model")
print(f"checkpoint: {STEPS_PATH}"
      f"{'  (exists - this run will resume)' if os.path.exists(STEPS_PATH) else ''}")


## 5. Server helpers

In [ ]:
# --- vLLM lifecycle: launch, wait until it truly answers, shut down. ---
import json, signal, socket, time, urllib.request

_json = json
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"


def _port_free(port=PORT):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def _cmd(model, extras=True):
    """Required args, plus tuning flags worth retrying without.

    Every optional flag has been renamed or dropped in some vLLM release, and a
    three-hour run should not die because a tuning knob moved.
    """
    required = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model["hf"],
        "--served-model-name", model["short"],
        "--host", "127.0.0.1", "--port", str(PORT),
        # T4 has no bfloat16; several of these configs request it by default.
        "--dtype", "float16",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEM_FRACTION),
        "--tensor-parallel-size", str(TP),
    ]
    return required + (["--max-num-seqs", str(MAX_NUM_SEQS), "--disable-log-requests"]
                       if extras else [])


def _post(path, payload, timeout=600):
    req = urllib.request.Request(
        f"{BASE_URL}{path}", data=_json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return _json.loads(r.read())


def start_server(model, timeout_s=2400, extras=True):
    """Ready means 'returned a completion', not '/health answered'.

    The endpoint accepts connections before weights finish loading, so health
    alone would let a run start early and fail every request at once.
    """
    if not _port_free():
        raise RuntimeError("port 8000 in use; run the shutdown cell")

    log_path = os.path.join(LOG_DIR, f"{model['short']}.log")
    log = open(log_path, "w")
    proc = subprocess.Popen(_cmd(model, extras), stdout=log,
                            stderr=subprocess.STDOUT, preexec_fn=os.setsid,
                            env=os.environ.copy())
    started = time.time()
    while True:
        if proc.poll() is not None:
            log.flush()
            tail = open(log_path).read()[-4000:]
            if extras and ("unrecognized arguments" in tail or "invalid choice" in tail):
                print("  optional flag rejected; retrying with required args only")
                return start_server(model, timeout_s, extras=False)
            raise RuntimeError(f"vLLM exited {proc.returncode}\n--- log tail ---\n{tail}")
        try:
            _post("/chat/completions", {"model": model["short"],
                                        "messages": [{"role": "user", "content": "ping"}],
                                        "max_tokens": 1}, timeout=20)
            print(f"  ready in {(time.time() - started) / 60:.1f} min")
            return proc, log_path
        except Exception:
            pass
        if time.time() - started > timeout_s:
            stop_server(proc)
            raise RuntimeError(f"not ready in {timeout_s}s\n{open(log_path).read()[-4000:]}")
        time.sleep(5)


def stop_server(proc):
    if proc is None or proc.poll() is not None:
        return
    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    try:
        proc.wait(timeout=90)
    except subprocess.TimeoutExpired:
        os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        proc.wait(timeout=30)
    time.sleep(5)   # let the GPUs actually free before the next load


def free_weights(model):
    """~15GB per model; the scratch disk does not hold four."""
    import shutil
    slug = "models--" + model["hf"].replace("/", "--")
    for root in (os.path.join(HF_CACHE, "hub"), HF_CACHE):
        p = os.path.join(root, slug)
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)


print("helpers ready")


## 6. Corpus

In [ ]:
# --- The corpus, in the one fixed order every model will walk. ---
# Shuffling once with a fixed seed and taking a prefix means any partial run is a
# uniform random sample rather than a biased slice of easy-first task ids, and it
# means the four models' task sets are nested rather than disjoint, so a
# cross-model comparison can be made paired on the tasks all of them reached.
import random

from datasets import load_dataset

from agentverif.harness import Task

ds = load_dataset("bigcode/bigcodebench", "default")
split = list(ds.keys())[0]
records = ds[split]

order = list(range(len(records)))
random.Random(SEED).shuffle(order)
TASKS = [Task.from_record(records[i]) for i in order[:N_TASKS]]

print(f"{len(records)} tasks in {split}; using {len(TASKS)} (seed {SEED})")
print("first five:", [t.task_id for t in TASKS[:5]])

# The reference solution must actually run here, or a failing test tells us
# nothing about the model. The corpus check measured 92%; this confirms the same
# environment before any GPU time is spent.
from agentverif.harness import run_tests

probe = [run_tests(t, t.reference_solution()) for t in TASKS[:12]]
ok = sum(r.passed for r in probe)
print(f"\nreference solutions passing: {ok}/12")
for t, r in zip(TASKS[:12], probe):
    if not r.passed:
        print(f"  {t.task_id}: {r.detail[:90]}")
if ok < 8:
    raise SystemExit(
        "Reference solutions are failing at a rate that would confound the "
        "correctness measurement. Stop and fix the environment first."
    )


## 7. Instrument check

In [ ]:
# --- Verify the instrument on this machine, before it is used to make a claim. ---
# The whole study rests on one property: a `# noqa` hides a finding from ruff and
# from nothing else, so divergence between ruff and bandit separates suppression
# from repair. That is a property of the installed tool versions, not a law, and
# it costs seconds to check rather than assume.
from dataclasses import asdict

from agentverif.analysers import analyse_all, suppressions_added
from agentverif.harness import write_source
from agentverif.transfer import fates_for_arm

VULNERABLE = (
    "import subprocess\n"
    "def run(cmd):\n"
    "    return subprocess.check_output(cmd, shell=True)\n"
)
SUPPRESSED = (
    "import subprocess  # noqa: S404\n"
    "def run(cmd):\n"
    "    return subprocess.check_output(cmd, shell=True)  # noqa: S602\n"
)
REPAIRED = (
    "import shlex\n"
    "import subprocess\n"
    "def run(cmd):\n"
    "    return subprocess.check_output(shlex.split(cmd))\n"
)


def counts(src):
    return {t: len(r) for t, r in analyse_all(write_source(src)).items()}


def as_step(src, arm, shown=""):
    res = analyse_all(write_source(src))
    return {"task_id": "instrument", "model": "check", "arm": arm, "round": 0,
            "shown_tool": shown,
            "findings": {t: [[f.code, f.line] for f in r.findings]
                         for t, r in res.items()}}


print(f"{'variant':14s} " + " ".join(f"{t:>8s}" for t in ("ruff", "bandit", "pylint")))
for name, src in (("vulnerable", VULNERABLE), ("suppressed", SUPPRESSED),
                  ("repaired", REPAIRED)):
    c = counts(src)
    print(f"{name:14s} " + " ".join(f"{c[t]:>8d}" for t in ("ruff", "bandit", "pylint")))

base = as_step(VULNERABLE, "baseline")
verdicts = {}
for name, src in (("suppressed", SUPPRESSED), ("repaired", REPAIRED)):
    fates = fates_for_arm(base, as_step(src, "shown_ruff", shown="ruff"))
    verdicts[name] = [f.transferred for f in fates if f.addressed]
    print(f"\n{name}: addressed {sum(f.addressed for f in fates)}/{len(fates)} "
          f"ruff findings, transfer verdicts {verdicts[name]}")

print("\ndirectives counted:", suppressions_added(VULNERABLE, SUPPRESSED))

# The two must land on opposite sides. If they do not, every number the study
# produces afterwards is uninterpretable, so this stops the notebook rather than
# letting a broken instrument spend three GPU hours.
ok = (verdicts["suppressed"] and not any(v for v in verdicts["suppressed"])
      and verdicts["repaired"] and all(verdicts["repaired"]))
print("\nINSTRUMENT", "OK - suppression and repair are distinguishable" if ok
      else "BROKEN")
if not ok:
    raise SystemExit(
        "The ruff/bandit pair no longer separates a `# noqa` from a real fix on "
        "these tool versions. Do not run the study until it does."
    )


## 8. Run

In [ ]:
# --- The sweep. One server at a time, hard time boxes, checkpointed to disk. ---
from agentverif.analysers import PYLINT_DISABLE, RUFF_SELECT
from agentverif.study import REPAIR_ROUNDS, run_study


def make_chat(model):
    """Adapt the OpenAI-compatible endpoint to the (reply, tokens) contract the
    study is written against. One retry: at 32 concurrent requests a transient
    failure would otherwise cost a whole task's record, and a retry costs a few
    seconds."""
    def chat(prompt):
        payload = {"model": model["short"],
                   "messages": [{"role": "user", "content": prompt}],
                   "temperature": TEMPERATURE, "max_tokens": MAX_GEN_TOKENS}
        last = None
        for attempt in range(2):
            try:
                r = _post("/chat/completions", payload, timeout=900)
                return (r["choices"][0]["message"]["content"] or "",
                        (r.get("usage") or {}).get("completion_tokens", 0))
            except Exception as exc:
                last = exc
                if attempt == 0:
                    time.sleep(3)
        raise last
    return chat


sweep_started = time.time()
runs = []

for i, model in enumerate(MODELS):
    elapsed = time.time() - sweep_started
    left = TOTAL_BUDGET_S - elapsed
    # Split what is left evenly across the models still to come, rather than
    # letting the first model spend the whole budget. Loading time comes out of
    # the same pot, so a slow download shortens its own model's run and not the
    # ones after it.
    budget = min(PER_MODEL_BUDGET_S, left / (len(MODELS) - i))

    print(f"\n{'=' * 72}\n[{i + 1}/{len(MODELS)}] {model['family']}  {model['hf']}")
    print(f"{elapsed / 60:.0f} min elapsed, {left / 60:.0f} min left, "
          f"this model gets up to {budget / 60:.0f} min of generation")

    if budget < MIN_USEFUL_BUDGET_S:
        print("  skipped: not enough budget left to produce a usable sample")
        runs.append({**model, "status": "skipped_no_budget", "tasks": 0})
        continue

    proc = None
    try:
        load_started = time.time()
        proc, log_path = start_server(model)
        load_min = (time.time() - load_started) / 60
        counters = run_study(TASKS, make_chat(model), model["short"], STEPS_PATH,
                            workers=WORKERS, time_budget_s=budget)
        runs.append({**model, "status": "ok", "load_min": round(load_min, 1),
                     **counters})
    except Exception as exc:
        print(f"  FAILED: {type(exc).__name__}: {exc}")
        runs.append({**model, "status": f"failed: {type(exc).__name__}", "tasks": 0})
    finally:
        stop_server(proc)
        free_weights(model)   # ~15GB each; the disk does not hold four

print(f"\n{'=' * 72}\nsweep finished in {(time.time() - sweep_started) / 3600:.2f}h")
print(f"{'model':24s} {'status':12s} {'load':>6s} {'tasks':>7s} {'steps':>7s} {'errors':>7s}")
for r in runs:
    print(f"{r['short']:24s} {r['status'][:12]:12s} {r.get('load_min', 0):>6} "
          f"{r.get('tasks', 0):>7} {r.get('steps', 0):>7} {r.get('errors', 0):>7}")

with open(os.path.join(OUT_DIR, "run_manifest.json"), "w") as fh:
    json.dump({"commit": COMMIT, "branch": BRANCH,
               "seed": SEED, "n_tasks": N_TASKS, "temperature": TEMPERATURE,
               "max_gen_tokens": MAX_GEN_TOKENS, "workers": WORKERS,
               "tensor_parallel": TP, "dtype": "float16",
               "analyser_versions": TOOL_VERSIONS,
               "ruff_select": RUFF_SELECT, "pylint_disable": PYLINT_DISABLE,
               "repair_rounds": REPAIR_ROUNDS,
               "runs": runs}, fh, indent=2)


## 9. Analysis

In [ ]:
# --- Analysis. Offline over the checkpoint, so it can be re-derived without a GPU. ---
from agentverif.report import (collect_fates, common_tasks, correctness_shift,
                               format_headline, headline, load_steps, restrict,
                               suppression_directives, traded_defects)

steps = load_steps(STEPS_PATH)
if not steps:
    # Every model failed or was skipped. Say so plainly rather than raising a
    # FileNotFoundError that reads like a bug in the analysis.
    raise SystemExit(
        f"No steps at {STEPS_PATH}. The sweep produced nothing - check the "
        f"status column in section 8 and the server logs in {LOG_DIR}."
    )

models = sorted({s["model"] for s in steps})
print(f"{len(steps)} steps, {len(models)} models: {', '.join(models)}")

n_by_model = {m: len({s["task_id"] for s in steps
                      if s["model"] == m and s["arm"] == "baseline"
                      and not s.get("error")})
              for m in models}
shared = common_tasks(steps)
print("tasks completed:", n_by_model)
print(f"shared by all models: {len(shared)}")

print("\n" + "=" * 96)
print("HEADLINE  of the findings an agent removed from the analyser it was shown,")
print("          how many were still reported by the held-out twin it never saw")
print("=" * 96)
print(format_headline(headline(steps)))

print("\npooled across models, on the tasks all of them reached")
print(format_headline(headline(restrict(steps, shared), by_model=False)))

print("\n" + "=" * 96)
print("CORRECTNESS  a finding removed by breaking the function is not a fix")
print("=" * 96)
print(f"{'model':24s} {'arm':14s} {'n':>5s} {'pass before':>12s} {'pass after':>11s} "
      f"{'broke':>7s} {'repaired':>9s}")
for r in correctness_shift(steps):
    print(f"{r['model']:24s} {r['arm']:14s} {r['n']:>5d} {r['pass_before']:>12d} "
          f"{r['pass_after']:>11d} {r['broke']:>7d} {r['repaired']:>9d}")

print("\n" + "=" * 96)
print("MECHANISM  suppression directives the agent actually wrote")
print("=" * 96)
rows = suppression_directives(steps)
if rows:
    tools = sorted({k for r in rows for k in r if k not in ("model", "arm")})
    print(f"{'model':24s} {'arm':14s} " + " ".join(f"{t:>9s}" for t in tools))
    for r in rows:
        print(f"{r['model']:24s} {r['arm']:14s} "
              + " ".join(f"{r.get(t, 0):>9d}" for t in tools))
else:
    print("none written")

print("\n" + "=" * 96)
print("TRADES  codes present after repair that were absent before")
print("=" * 96)
trades = traded_defects(steps)
for code, n in list(trades.items())[:25]:
    print(f"  {code:24s} {n:>5d}")
if not trades:
    print("  none")

# Everything the write-up needs, saved next to the raw steps so the analysis can
# be redone or re-sliced by severity without another GPU hour.
summary = {
    "n_steps": len(steps),
    "tasks_by_model": n_by_model,
    "shared_tasks": sorted(shared),
    "headline_by_model": headline(steps),
    "headline_pooled_shared": headline(restrict(steps, shared), by_model=False),
    "correctness": correctness_shift(steps),
    "directives": suppression_directives(steps),
    "trades": trades,
    "fates": [vars(f) | {"transferred": f.transferred} for f in collect_fates(steps)],
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(f"\nwrote {OUT_DIR}/summary.json and {STEPS_PATH}")
print("Download the whole results/ folder from the notebook output before the "
      "session expires.")


## 10. Emergency shutdown (only if needed)

In [ ]:
# --- Emergency shutdown, if a cell was interrupted and the port is stuck. ---
subprocess.run("pkill -f vllm.entrypoints.openai.api_server", shell=True)
time.sleep(5)
print("port free:", _port_free())


## Reading the result

The headline is one number per model: **of the findings the agent removed from the
analyser it was shown, how many were still reported by the held-out twin.**

| result | what it would mean |
|---|---|
| high, with the interval clear of 50% | agents satisfy the detector rather than the code, which is the empirical case for verification that is independent of the tool being optimised against |
| low, interval clear of 50% | quality gates generalise; a fix aimed at one analyser is a real fix. Good news, and as far as we can tell unmeasured |
| interval spanning 50% | the sample is too small to say. Re-run: the notebook resumes and the study grows |

Either of the first two is publishable, which is the property the design was chosen
for. The third is a statement about sample size, not about agents, and the write-up
has to say so rather than reporting the point estimate as though it settled anything.

## What is deliberately not claimed

**The pylint arm reports no transfer rate.** Pylint's message ids have no ruff or
bandit counterpart, so that arm can say what was *addressed* but not whether the fix
*transferred*. It is reported as unmeasurable rather than as zero suppression.
What the pylint arm does contribute is direct: the `# pylint: disable` directives the
agent wrote, whether repair broke working code, and which new defects appeared.

**One sample per task at temperature 0.** The variance budget went into tasks rather
than into seeds, because the estimate is over findings and more tasks tighten it
faster than more samples of the same task.

**Findings on generated Python from one benchmark.** BigCodeBench is
library-heavy single-file code. Nothing here extends to Java, to CodeQL, or to
repository-scale change without being measured there too.

## Outputs

| file | contents |
|---|---|
| `results/steps.jsonl` | every step of every arm: findings by tool and code, test result, directives, tokens |
| `results/summary.json` | every table above, plus the per-finding fates behind them |
| `results/run_manifest.json` | models, seed, budgets, analyser versions |

`steps.jsonl` is the raw record. All of the analysis re-derives from it offline via
`agentverif.report`, so the tables can be re-sliced by severity or corrected without
another GPU hour.
